<a href="https://colab.research.google.com/github/kbrh3/Gen_AI_Class/blob/main/KailynnB_hw_tokenization_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Homework 1 — Tokenization and Embeddings



In [ ]:
#chunk 1

import re #regular expressions
import random #random seed
from collections import Counter #count token frequency

import torch #pytorch tensor library
import torch.nn as nn #CHANGE: like old project, explicit nn import

import tiktoken #openai gpt style tokenizer

#seeds for reproducibility
random.seed(42)
torch.manual_seed(42)

with open("/content/the-verdict.txt", "r", encoding="utf-8") as f:
    text = f.read()

#required output
print("Character count:", len(text))
print("Line count:", text.count("\n") + 1)
print("\nFirst 300 characters preview:\n")
print(text[:300])


Character count: 20479
Line count: 165

First 300 characters preview:

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would ha


## Part 1 — Regex Tokenization (Simple Tokenizer)


In [ ]:
#chunk 2

def simple_tokenizer(text: str):
    #return simple_tokenizer(text)
    #words OR single punctuation char
    pattern = r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?|[^\w\s]"
    tokens = re.findall(pattern, text)
    return tokens

#tokenize full corpus
tokens = simple_tokenizer(text)

#quick check
print("First 60 tokens:\n", tokens[:60])

sample = "cheap genius--though a good fellow enough--so"
print("\nExample showing punctuation split:")
print("sample:", sample)
print("tokens:", simple_tokenizer(sample))


First 60 tokens:
 ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '-', '-', 'though', 'a', 'good', 'fellow', 'enough', '-', '-', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(']

Example showing punctuation split:
sample: cheap genius--though a good fellow enough--so
tokens: ['cheap', 'genius', '-', '-', 'though', 'a', 'good', 'fellow', 'enough', '-', '-', 'so']


### Token Statistics and Analysis


In [ ]:
#chunk 3
total_tokens = len(tokens)
unique_tokens = len(set(tokens))

print("Total token count is ", total_tokens)
print("Unique token count is ", unique_tokens)
token_counts = Counter(tokens)

print("\nTop 30 tokens by frequency:")
for tok, count in token_counts.most_common(30):
    print(f"{tok} : {count}")

#punctuation-like = only non-word chars
#word-like = contains at least one word char
punct_tokens = [t for t in tokens if re.fullmatch(r"[^\w]+", t)]
word_tokens = [t for t in tokens if re.search(r"\w", t)]

punct_frac = len(punct_tokens) / total_tokens
word_frac = len(word_tokens) / total_tokens

print("\nPunctuation-like token fraction ", punct_frac)
print("Word-like token fraction ", word_frac)


Total token count is  4669
Unique token count is  1155

Top 30 tokens by frequency:
- : 232
, : 229
. : 212
the : 168
" : 137
I : 117
of : 98
to : 97
a : 82
and : 75
was : 71
it : 65
his : 61
he : 60
that : 57
had : 53
in : 40
my : 40
me : 37
him : 33
with : 32
on : 27
have : 26
as : 24
? : 24
! : 24
one : 23
you : 23
Mrs : 22
; : 22

Punctuation-like token fraction: 0.1974726922253159
Word-like token fraction: 0.8025273077746841


## Part 2 — Vocabulary, Encode/Decode, and Special Tokens


In [ ]:
#chunk 4
SPECIAL_TOKENS = ["<|unk|>", "<|endoftext|>", "[PAD]"]
def build_vocab(tokens, special_tokens=SPECIAL_TOKENS):
    vocab = {}
    for tok in special_tokens:
        vocab[tok] = len(vocab)
    uniq = sorted(set(tokens))
    for tok in uniq:
        if tok not in vocab:
            vocab[tok] = len(vocab)

    inverse_vocab = {i: tok for tok, i in vocab.items()}
    return vocab, inverse_vocab

vocab, inv_vocab = build_vocab(tokens)
print("Vocab size:", len(vocab))
print("Special token ids:")
for tok in SPECIAL_TOKENS:
    print(tok, "->", vocab[tok])

print("\nExample token ids:")
for t in ["the", ",", "I"]:
    print(t, "->", vocab.get(t, vocab["<|unk|>"]))



Vocab size: 1158
Special token ids:
<|unk|> -> 0
<|endoftext|> -> 1
[PAD] -> 2

Example token ids:
the -> 1012
, -> 8
I -> 59


### Encode/Decode Correctness Tests


In [ ]:
#chunk 5
def encode(tokens, vocab):
    #convert tokens to ids, use <|unk|> for unseen tokens
    return [vocab.get(t, vocab["<|unk|>"]) for t in tokens]

def decode(ids, inv_vocab):
    #convert ids back to tokens
    #spacing does not need to be perfect
    return " ".join(inv_vocab[i] for i in ids)

s = text[:200]
rt_tokens = regex_tokenize(s)
rt_ids = encode(rt_tokens, vocab)
rt_decoded = decode(rt_ids, inv_vocab)

print("Round-trip test:")
print(rt_decoded)

oov_sentence = "flibbertigibbet snickerdoodle quantumflux zappertron blorf"
oov_tokens = regex_tokenize(oov_sentence)
oov_ids = encode(oov_tokens, vocab)

print("\nOOV test:")
print("tokens:", oov_tokens)
print("ids:", oov_ids)
print("decoded:", decode(oov_ids, inv_vocab))

end_tokens = regex_tokenize("This is the end")
end_tokens.append("<|endoftext|>")
end_ids = encode(end_tokens, vocab)

print("\nEnd token test:")
print("tokens:", end_tokens)
print("decoded:", decode(end_ids, inv_vocab))


Round-trip test:
I HAD always thought Jack Gisburn rather a cheap genius - - though a good fellow enough - - so it was no great surprise to me to hear that , in the height of his glory , he had dropped his painting , married a

OOV test:
tokens: ['flibbertigibbet', 'snickerdoodle', 'quantumflux', 'zappertron', 'blorf']
ids: [0, 0, 0, 0, 0]
decoded: <|unk|> <|unk|> <|unk|> <|unk|> <|unk|>

End token test:
tokens: ['This', 'is', 'the', 'end', '<|endoftext|>']
decoded: This is the end <|endoftext|>


## Part 3 — BPE Tokenization with tiktoken (GPT-2)


In [ ]:
#chunk 6
gpt2_tokenizer = tiktoken.get_encoding("gpt2")
bpe_ids = gpt2_tokenizer.encode(text)

print("Total BPE token count ", len(bpe_ids))
print("Unique BPE token ids ", len(set(bpe_ids)))
begin_excerpt = text[:300]
mid_start = len(text) // 2
mid_excerpt = text[mid_start: mid_start + 300]
end_excerpt = text[-300:]

excerpts = {
    "beginning": begin_excerpt,
    "middle": mid_excerpt,
    "end": end_excerpt
}

for name, excerpt in excerpts.items():
    regex_count = len(simple_tokenizer(excerpt))
    bpe_count = len(gpt2_tokenizer.encode(excerpt))
    round_trip = gpt2_tokenizer.decode(gpt2_tokenizer.encode(excerpt))

    print(f"\n--- {name.upper()} EXCERPT ---")
    print("chars ", len(excerpt))
    print("regex token count ", regex_count)
    print("BPE token count ", bpe_count)
    print("\ntiktoken round-trip preview \n")
    print(round_trip[:300])


Total BPE token count  5145
Unique BPE token ids  1416

--- BEGINNING EXCERPT ---
chars  300
regex token count  67
BPE token count  70

tiktoken round-trip preview 

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would ha

--- MIDDLE EXCERPT ---
chars  300
regex token count  77
BPE token count  79

tiktoken round-trip preview 

hout that."

Well--it was just the end one might have foreseen for him. Only he was, through it all and in spite of it all--as he had been through, and in spite of, his pictures--so handsome, so charming, so disarming, that one longed to cry out: "Be dissatisfied with your leisure!" as once one had 

--- END EXCERPT ---
chars  300
regex token count  70
BPE token count  76

tiktoken round-trip preview 

 brings me anywhere

## Part 4 — Sliding-Window Dataset for Next-Token Prediction


In [ ]:
#chunk 7
def sliding_windows(ids, context_len=32, stride=16):
    xs, ys = [], []
    for start in range(0, len(ids) - context_len - 1, stride):
        x = ids[start : start + context_len]
        y = ids[start + 1 : start + context_len + 1]
        xs.append(x)
        ys.append(y)
    return xs, ys

xs, ys = sliding_windows(bpe_ids, context_len=32, stride=16)

print("Number of windows:", len(xs))
print("x window length:", len(xs[0]))
print("y window length:", len(ys[0]))

print("\nDecoded x (context):\n")
print(gpt2_tokenizer.decode(xs[0]))

print("\nDecoded y (targets):\n")
print(gpt2_tokenizer.decode(ys[0]))



Number of windows: 320
x window length: 32
y window length: 32

Decoded x (context):

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that,

Decoded y (targets):

 HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in


## Part 5 — Token and Positional Embeddings


In [ ]:
#chunk 8
import torch.nn as nn
V = gpt2_tokenizer.n_vocab   #GPT-2 vocab size
D = 64                       #embedding dimension
T = 32                       #context length

print("Vocab size (V) ", V)
print("Embedding dim (D) ", D)
print("Context length (T) ", T)

#create embedding layers
token_emb = nn.Embedding(V, D)
pos_emb = nn.Embedding(T, D)

#create a small batch from sliding windows
#use first 2 windows
x_batch = torch.tensor(xs[:2], dtype=torch.long)
print("\nx_batch shape:", x_batch.shape)

#compute embeddings
tok = token_emb(x_batch)           #[B, T, D]
pos = pos_emb(torch.arange(T))     #[T, D]
inp = tok + pos                    #[B, T, D]

print("token embedding shape ", tok.shape)
print("positional embedding shape ", pos.shape)
print("final input embedding shape ", inp.shape)
#same token - diff demo
same_id = x_batch[0, 0].item()

print("\nUsing same token id ", same_id)
print("Decoded token ", gpt2_tokenizer.decode([same_id]))

#token embedding should be identical
print("\nToken embedding (same id, checked twice) ")
print(token_emb(torch.tensor(same_id))[:5])
print(token_emb(torch.tensor(same_id))[:5])
#force same token at two positions
x_forced = x_batch.clone()
x_forced[0, 1] = same_id

tok_forced = token_emb(x_forced)
inp_forced = tok_forced + pos

print("\nFinal embedding at position 0 (first 5 values) ")
print(inp_forced[0, 0][:5])
print("\nFinal embedding at position 1 (first 5 values)")
print(inp_forced[0, 1][:5])


Vocab size (V)  50257
Embedding dim (D)  64
Context length (T)  32

x_batch shape: torch.Size([2, 32])
token embedding shape  torch.Size([2, 32, 64])
positional embedding shape  torch.Size([32, 64])
final input embedding shape  torch.Size([2, 32, 64])

Using same token id  40
Decoded token  I

Token embedding (same id, checked twice) 
tensor([-0.4329, -0.4930, -0.8305,  0.7459, -0.3704], grad_fn=<SliceBackward0>)
tensor([-0.4329, -0.4930, -0.8305,  0.7459, -0.3704], grad_fn=<SliceBackward0>)

Final embedding at position 0 (first 5 values) 
tensor([ 1.3610, -1.5991, -1.0349, -0.4879, -0.2453], grad_fn=<SliceBackward0>)

Final embedding at position 1 (first 5 values)
tensor([-2.0984,  0.5057,  0.0099, -0.2243, -0.2063], grad_fn=<SliceBackward0>)
